# Анализ эксперимента с NeuralControllerEnv

In [ ]:
import matplotlib.pyplot as plt

from nn_laser_stabilizer.config.config import load_config
from nn_laser_stabilizer.utils.paths import get_experiment_dir

EXPERIMENT_NAME = "neural_controller"
EXPERIMENT_DATE = "2026-01-24"
EXPERIMENT_TIME = "16-36-00"

EXPERIMENT_DIR_PATH = get_experiment_dir(
    experiment_name=EXPERIMENT_NAME,
    experiment_date=EXPERIMENT_DATE,
    experiment_time=EXPERIMENT_TIME,
)

CONFIG_PATH = EXPERIMENT_DIR_PATH / "config.yaml"
config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")

## Анализ логов окружения

In [ ]:
import numpy as np
import pandas as pd

from nn_laser_stabilizer.analysis.sources import parse_keyval_log

In [ ]:
ENV_LOG_PATH = EXPERIMENT_DIR_PATH / "env_logs" / "env.log"

# [ENV] step: / [ENV] reset: — различаем по event; reset наследует номер шага.
_raw = parse_keyval_log(ENV_LOG_PATH)
_env = _raw[_raw["tag"] == "ENV"].reset_index(drop=True)
_is_step = _env["event"] == "step"

# step_interval (в логе с суффиксом 'us') есть только у шагов; отсутствующее → 0.0,
# у reset-строк → NaN (как в прежнем парсере).
_step_interval = _env["step_interval"].fillna(0.0) if "step_interval" in _env.columns else 0.0

env_df = pd.DataFrame({
    "Step": _env["step"],
    "Type": np.where(_is_step, "step", "reset"),
    "Process variable": _env["process_variable"],
    "Setpoint": _env["setpoint"],
    "Error": _env["error"],
    "Action": _env["action"].where(_is_step),
    "Control output": _env["control_output"],
    "Reward": _env["reward"].where(_is_step),
    "Step interval (us)": np.where(_is_step, _step_interval, np.nan),
})
env_df["Step"] = env_df["Step"].ffill().fillna(0).astype(int)
print(f"Загружено {len(env_df)} записей из логов окружения")

In [ ]:
step_df = env_df[env_df['Type'] == 'step'].copy()
exploration_steps = config.exploration.steps

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(step_df['Step'], step_df['Step interval (us)'], alpha=0.8, linewidth=0.8)
plt.title('Step interval over steps')
plt.xlabel('Step')
plt.ylabel('Step interval (us)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(step_df['Step'], step_df['Error'], alpha=0.8, linewidth=0.8, label='Error')

plt.axvline(x=exploration_steps, color='red', linestyle='--', linewidth=2, 
            label=f'Switch to NN (step {exploration_steps})')

plt.title(f'Error over Steps')
plt.xlabel('Step')
plt.ylabel('Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(step_df['Step'], step_df['Reward'], alpha=0.8, linewidth=0.8, label='Reward')

plt.axvline(x=exploration_steps, color='red', linestyle='--', linewidth=2, 
            label=f'Switch to NN (step {exploration_steps})')

plt.title('Reward over Steps')
plt.xlabel('Step')
plt.ylabel('Reward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(step_df['Step'], step_df['Process variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')
plt.plot(step_df['Step'], step_df['Setpoint'], color='r', linestyle='--', linewidth=1.5, label='Setpoint')

plt.axvline(x=exploration_steps, color='red', linestyle='--', linewidth=2, 
            label=f'Switch to NN (step {exploration_steps})')

plt.title('Process Variable over Steps')
plt.xlabel('Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(step_df['Step'], step_df['Control output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')
plt.xlabel('Step', fontsize=12)
plt.ylabel('Control Output', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc='best')

plt.axvline(x=exploration_steps, color='red', linestyle='--', linewidth=2, 
            label=f'Switch to NN (step {exploration_steps})')

plt.title('Control Output over Steps')
plt.tight_layout()
plt.show()

## Анализ процесса обучения

In [ ]:
from nn_laser_stabilizer.analysis.sources import parse_train_log

In [ ]:
TRAIN_LOG_PATH = EXPERIMENT_DIR_PATH / "train_logs" / "train.log"

# [TRAIN] step: [actor_loss] buffer_size loss_q1 loss_q2 step time.
loss_df = parse_train_log(TRAIN_LOG_PATH)
loss_df = (
    loss_df[loss_df[["loss_q1", "loss_q2", "buffer_size"]].notna().all(axis=1)]
    .reindex(columns=["step", "loss_q1", "loss_q2", "actor_loss", "buffer_size"])
    .reset_index(drop=True)
)
print(f"Загружено {len(loss_df)} записей из логов обучения")
print(f"Диапазон шагов обучения: {loss_df['step'].min()} - {loss_df['step'].max()}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(loss_df['step'], loss_df['loss_q1'], 'b-', alpha=0.7, label='Q1 Loss')
axes[0].set_title('Q1 Loss')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss_df['step'], loss_df['loss_q2'], 'g-', alpha=0.7, label='Q2 Loss')
axes[1].set_title('Q2 Loss')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_df['step'], loss_df['loss_q1'] + loss_df['loss_q2'], 'r--', alpha=0.7, label='Sum (Q1 + Q2)')
axes[2].set_title('Sum (Q1 + Q2)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
actor_loss_df = loss_df[loss_df['actor_loss'].notna()]

plt.figure(figsize=(12, 5))

plt.plot(actor_loss_df['step'], actor_loss_df['actor_loss'], 'r-', alpha=0.7)
plt.title('Actor Loss')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(loss_df['step'], loss_df['buffer_size'], 'm-', alpha=0.7)
plt.title('Buffer Size')
plt.xlabel('Step')
plt.ylabel('Size')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()